# 05 — Histograms

Two kinds of histograms:

1. **Crossmatch quality**: real vs. shifted-false AllWISE matches to the raw DESI/QSO_Sample
   catalogs, to sanity-check the matching radius against the false-match rate. Independent of
   the matching/SED/color-color pipeline — only needs `data/raw/`.
2. **UV-excess E(B-V) distribution**: E(B-V) histograms and UV-excess fraction bar chart for the
   combined DESI+W2M sample, split by UV-excess flag.

**Script references:** `scripts/histogram.py`, `histogram2.py`,
`scripts/colorcolor/ebv_uv_excess_histogram.py`, `ebv_uv_excess_fraction_barchart.py`

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import yaml
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.table import Table
from astroquery.xmatch import XMatch
from synphot import units as su

BASE_DIR = Path.cwd().parent
with open(BASE_DIR / "config" / "qso_params.yaml") as f:
    PARAMS = yaml.safe_load(f)

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_MATCHED = BASE_DIR / "data" / "matched"
WAVELENGTHS = PARAMS["band_wavelengths_AA"]
FLAM_MAX_GALEX_PS1 = PARAMS["flux_outlier_thresholds_flam"]["galex_panstarrs"] * su.FLAM
EBV_MIN = PARAMS["uv_excess"]["ebv_min"]


def mag_arr(col):
    if hasattr(col, 'filled'):
        return np.array(col.filled(np.nan), dtype=float)
    arr = np.array(col, dtype=str)
    arr[arr == '--'] = 'nan'
    return arr.astype(float)

## Crossmatch quality: real vs. false AllWISE matches

Source: `scripts/histogram.py` (DESI full catalog), `histogram2.py` (QSO_Sample subset).
Shifts DEC by +5 deg to build a false-match control sample, then compares angular separation
distributions out to 12 arcsec against the current AllWISE catalog. Useful for sanity-checking
the 2 arcsec matching radius (`config/qso_params.yaml`'s `crossmatch.radius_arcsec`) against
where the false-match rate starts to dominate.

In [ ]:
def plot_match_quality(csv_path, ra_col='RA', dec_col='DEC', target_catalog='II/311/wise',
                        max_radius_arcsec=12, title_suffix=''):
    real_catalog = Table.read(csv_path, format='csv')
    false_catalog = real_catalog.copy()
    false_catalog[dec_col] = false_catalog[dec_col] + 5

    max_radius = max_radius_arcsec * u.arcsec
    real_matches = XMatch.query(cat1=real_catalog, cat2=target_catalog, max_distance=max_radius,
                                 colRA1=ra_col, colDec1=dec_col)
    false_matches = XMatch.query(cat1=false_catalog, cat2=target_catalog, max_distance=max_radius,
                                  colRA1=ra_col, colDec1=dec_col)

    real_separations = np.array(real_matches["angDist"])
    false_separations = np.array(false_matches["angDist"])
    bin_edges = np.arange(0, max_radius_arcsec + 0.25, 0.25)

    rCounts, _ = np.histogram(real_separations, bins=bin_edges)
    fCounts, _ = np.histogram(false_separations, bins=bin_edges)
    diff_counts = rCounts - fCounts

    fig, ax = plt.subplots(figsize=(8, 5.5))
    ax.stairs(rCounts, bin_edges, baseline=0, linewidth=2, color="black", label="Matches")
    ax.stairs(fCounts, bin_edges, baseline=0, fill=True, color="red", alpha=0.3,
              label="Shifted (False) matches")
    ax.stairs(diff_counts, bin_edges, baseline=0, linewidth=2, color="blue", alpha=0.5,
              label="Difference (Real - False)")

    ax.axvline(PARAMS["crossmatch"]["radius_arcsec"], color='green', linestyle='--',
               label=f'current radius ({PARAMS["crossmatch"]["radius_arcsec"]}\")')
    ax.set_xlim(0, max_radius_arcsec)
    ax.set_xlabel("Separation (arcsec)", fontsize=12)
    ax.set_ylabel("Number", fontsize=12)
    ax.grid(linestyle=":", alpha=0.5)
    ax.legend(loc="upper right")
    plt.title(f"{target_catalog} and DESI real and false matches{title_suffix}")
    plt.tight_layout()
    plt.show()


# Full DESI catalog
plot_match_quality(DATA_RAW / "COMBINED_QSOS_TAB.csv", title_suffix=' (full catalog)')

# QSO_Sample subset
plot_match_quality(DATA_RAW / "QSO_Sample.csv", title_suffix=' (QSO_Sample)')

## E(B-V) distribution: UV-excess vs. non-UV-excess

Source: `scripts/colorcolor/ebv_uv_excess_histogram.py`. DESI rows use `EBV`/`gmag`/`rmag`;
W2M rows (`src == 'w2m'`) use their own lowercase `ebv`/`gmag_2`/`rmag_2` and skip the E(B-V)
cut (W2M's E(B-V) runs much lower than DESI's).

In [ ]:
def uv_excess_mask(table, gmag_col, rmag_col, require_ebv=True, ebv_col='EBV'):
    lam_fuv = WAVELENGTHS['FUV'] * u.AA
    lam_nuv = WAVELENGTHS['NUV'] * u.AA
    lam_g = WAVELENGTHS['g'] * u.AA
    lam_r = WAVELENGTHS['r'] * u.AA

    flam_fuv = (mag_arr(table['FUVmag']) * u.ABmag).to(su.FLAM, u.spectral_density(lam_fuv))
    flam_fuv[flam_fuv > FLAM_MAX_GALEX_PS1] = np.nan
    flam_nuv = (mag_arr(table['NUVmag']) * u.ABmag).to(su.FLAM, u.spectral_density(lam_nuv))
    flam_nuv[flam_nuv > FLAM_MAX_GALEX_PS1] = np.nan
    flam_g = (mag_arr(table[gmag_col]) * u.ABmag).to(su.FLAM, u.spectral_density(lam_g))
    flam_g[flam_g > FLAM_MAX_GALEX_PS1] = np.nan
    flam_r = (mag_arr(table[rmag_col]) * u.ABmag).to(su.FLAM, u.spectral_density(lam_r))
    flam_r[flam_r > FLAM_MAX_GALEX_PS1] = np.nan

    f_fn = flam_fuv / flam_nuv
    f_ng = flam_nuv / flam_g
    f_gr = flam_g / flam_r

    flux_criterion = ((f_ng > 1) & (f_gr < 1)) | ((f_fn > 1) & (f_ng < 1))

    if require_ebv:
        ebv = mag_arr(table[ebv_col])
        return flux_criterion & (ebv > EBV_MIN)
    return flux_criterion


table = Table.read(DATA_MATCHED / "FINAL_COMBINED_QSOs_W2M.csv", format='csv')

src = np.array(table['src'], dtype=str)
is_desi = src != 'w2m'
is_w2m = ~is_desi

uv_flag = np.zeros(len(table), dtype=bool)
if np.any(is_desi):
    uv_flag[is_desi] = uv_excess_mask(table[is_desi], gmag_col='gmag', rmag_col='rmag', require_ebv=True)
if np.any(is_w2m):
    uv_flag[is_w2m] = uv_excess_mask(table[is_w2m], gmag_col='gmag_2', rmag_col='rmag_2', require_ebv=False)

ebv_desi_all = mag_arr(table['EBV'])
ebv_w2m_all = mag_arr(table['ebv'])

ebv_desi = ebv_desi_all[is_desi]
uv_flag_desi = uv_flag[is_desi]
ebv_w2m = ebv_w2m_all[is_w2m]
uv_flag_w2m = uv_flag[is_w2m]


def plot_ebv_histogram(ax, ebv, flag, title):
    bin_edges = np.arange(0, 2 + 0.1, 0.1)
    ax.hist(ebv[~flag], bins=bin_edges, histtype='stepfilled', color='steelblue', alpha=0.7,
            label='No UV excess')
    ax.hist(ebv[flag], bins=bin_edges, histtype='stepfilled', color='darkorange', alpha=0.9,
            label='UV excess')
    ax.set_title(title)
    ax.set_xlabel('E(B-V)')
    ax.set_ylabel('Number of QSOs')
    ax.legend(fontsize=11)
    ax.set_yscale('log')


fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
ebv_combined = np.concatenate([ebv_desi, ebv_w2m])
uv_flag_combined = np.concatenate([uv_flag_desi, uv_flag_w2m])
plot_ebv_histogram(axes[0], ebv_combined, uv_flag_combined, 'DESI + W2M combined sample')
plot_ebv_histogram(axes[1], ebv_desi, uv_flag_desi, 'DESI sample only')
plot_ebv_histogram(axes[2], ebv_w2m, uv_flag_w2m, 'W2M sample only')
fig.suptitle('E(B-V) distribution of UV-excess vs. non-UV-excess QSOs')
fig.tight_layout()
plt.show()

## UV-excess fraction per E(B-V) bin

Source: `scripts/colorcolor/ebv_uv_excess_fraction_barchart.py`. Reuses the `uv_flag`/`ebv`
arrays computed above.

In [ ]:
bin_edges = np.arange(0, 2 + 0.1, 0.1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

total_counts, _ = np.histogram(ebv_combined, bins=bin_edges)
excess_counts, _ = np.histogram(ebv_combined[uv_flag_combined], bins=bin_edges)

with np.errstate(invalid='ignore', divide='ignore'):
    fraction = np.where(total_counts > 0, excess_counts / total_counts, np.nan)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.bar(bin_centers, fraction, width=0.09, color='darkorange', edgecolor='black')
ax.set_xlabel('E(B-V)')
ax.set_ylabel('Fraction of QSOs with UV excess')
ax.set_title('UV-excess fraction per E(B-V) bin (DESI + W2M combined sample)')
ax.set_ylim(0, 1)
fig.tight_layout()
plt.show()